# Rooting by branch lengths

When no outgroup is available, `toytree` can infer a root position from edge lengths. The three public methods are `root_on_midpoint()`, `root_on_balanced_midpoint()`, and `root_on_minimal_ancestor_deviation()`. All three operate on the current unrooted topology, but they make different assumptions about rate variation.

Use midpoint and balanced midpoint as simple clock-like heuristics. Use MAD when branch-length variation may reflect rate heterogeneity and you want edge-wise scores for alternative roots.


In [ ]:
import toytree
from loguru import logger

logger.remove()


## Compare the three branch-length methods

This non-ultrametric example is rooted three different ways on the same unrooted topology.


In [ ]:
base = toytree.tree("(((a:1,b:1):2,c:4):1,(d:1,e:5):1);")
utree = base.unroot()

mid = utree.mod.root_on_midpoint()
bal = utree.mod.root_on_balanced_midpoint()
mad = utree.mod.root_on_minimal_ancestor_deviation()

c, a, m = toytree.mtree([mid, bal, mad]).draw(
    layout="d",
    node_labels="idx",
    node_sizes=12,
    tip_labels=True,
    width=720,
    height=260,
)
a[0].label.text = "midpoint"
a[1].label.text = "balanced midpoint"
a[2].label.text = "MAD"
c


## Midpoint rooting

`tree.mod.root_on_midpoint()` places the root at the midpoint of the longest tip-to-tip path. It is fast and often useful as a first pass, but it assumes a clock-like interpretation of branch lengths.

Reference: Farris, J. S. Estimating phylogenetic trees from distance matrices. *The American Naturalist* 106, 645-668 (1972).


In [ ]:
utree.mod.root_on_midpoint().draw(layout="d", node_labels="idx", node_sizes=14);


## Balanced midpoint rooting

`tree.mod.root_on_balanced_midpoint()` solves a related tree-center problem by minimizing the maximum distance from the root to the tips. It is still a clock-like heuristic, but is often less sensitive than simple midpoint rooting to long outlier branches.

The `tolerance` argument controls the stopping criterion when optimizing the position on the chosen edge.


In [ ]:
utree.mod.root_on_balanced_midpoint(tolerance=1e-8).draw(
    layout="d",
    node_labels="idx",
    node_sizes=14,
);


## Minimal ancestor deviation (MAD)

`tree.mod.root_on_minimal_ancestor_deviation()` evaluates every rootable edge and chooses the edge and position that minimize the global ancestor-deviation score. It is the most informative branch-length rooting method in `toytree` because it also stores edge-wise support for alternative root positions.

Reference: Tria, F., Landan, G. & Dagan, T. Phylogenetic rooting using minimal ancestor deviation. *Nature Ecology & Evolution* 1, 0193 (2017).


In [ ]:
# example adapted from the rooting tests so the returned statistics are stable
madtree = (
    toytree.rtree.rtree(5, seed=123)
    .unroot()
    .set_node_data("name", {i: j for i, j in enumerate("abcdeXYR")})
    .set_node_data("dist", {"e": 5, "Y": 3})
)

mad_rooted, stats = madtree.mod.root_on_minimal_ancestor_deviation(return_stats=True)
stats


`return_stats=True` returns a rooted tree plus a summary dictionary. The three main values are:

- `minimal_ancestor_deviation`: the best MAD score; lower is better.
- `root_ambiguity_index`: how close the next-best root is to the best root; lower means a clearer optimum.
- `root_clock_coefficient_of_variation`: how far the chosen rooting is from a strict clock interpretation.


## Constrain MAD to a chosen edge with `query`

If you want to evaluate a user-chosen root edge rather than the global optimum, pass a node query. MAD will still optimize the position along that edge and return the corresponding statistics.


In [ ]:
mad_opt, opt_stats = madtree.mod.root_on_minimal_ancestor_deviation(return_stats=True)
mad_alt, alt_stats = madtree.mod.root_on_minimal_ancestor_deviation("a", return_stats=True)

c, a, m = toytree.mtree([mad_opt, mad_alt]).draw(
    layout="d",
    node_labels="idx",
    node_sizes=12,
    tip_labels=True,
    width=520,
    height=240,
)
a[0].label.text = f"optimal MAD (CV={opt_stats['root_clock_coefficient_of_variation']:.1f})"
a[1].label.text = f"root constrained to a (CV={alt_stats['root_clock_coefficient_of_variation']:.1f})"
c


## Plot `MAD` and `MAD_root_prob` on branches

The returned rooted tree stores two edge features automatically:

- `MAD`: the deviation score if the tree were rooted on that edge.
- `MAD_root_prob`: the relative support for each possible root edge.


In [ ]:
mad_rooted.get_node_data()[["name", "MAD", "MAD_root_prob"]]


In [ ]:
c, a, m = mad_rooted.draw(layout="d", width=480, node_sizes=10, node_labels="idx")
mad_rooted.annotate.add_edge_labels(a, "MAD_root_prob", mask=False, font_size=11)
c


## Related APIs

- [`mod-rooting-outgroup.md`](mod-rooting-outgroup.md) for manual outgroup rooting.
- [`mod-rooting-dlc.md`](mod-rooting-dlc.md) for reconciliation-based rooting of gene trees.
